In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
# ======
com_dir = "your_directory/loads/com/"
res_dir = "your_directory/loads/res/"
output_dir = "your_directory/loads/new/"
os.makedirs(output_dir, exist_ok=True)
efs_csv = "your_directory/loads/EFSLoadProfile_Reference_Moderate.csv"
fips_npz = "your_directory/loads/fips_population_2018.npz"
# ======
efs_df = pd.read_csv(efs_csv)
efs_df = efs_df[(efs_df["Year"] == 2018)]
efs_df["LocalHourID"] = efs_df["LocalHourID"].astype(int)
efs_df["LoadMW"] = pd.to_numeric(efs_df["LoadMW"], errors="coerce").fillna(0)
# ======
fips_data = np.load(fips_npz, allow_pickle=True)
fips_list = fips_data["FIPS"]
pop_list = fips_data["POP2018"]
fips_pop_dict = dict(zip(fips_list, pop_list))
state_pop = {}
for fips, pop in fips_pop_dict.items():
    state = fips[:2]
    state_pop[state] = state_pop.get(state, 0) + pop

In [ ]:
fips2abbr = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA",
    "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL",
    "13": "GA", "15": "HI", "16": "ID", "17": "IL", "18": "IN",
    "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME",
    "24": "MD", "25": "MA", "26": "MI", "27": "MN", "28": "MS",
    "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI",
    "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT",
    "50": "VT", "51": "VA", "53": "WA", "54": "WV", "55": "WI",
    "56": "WY"
}
# === UTC time difference ===
state_to_utc_offset = {
    "AL": -6, "AK": -9, "AZ": -7, "AR": -6, "CA": -8,
    "CO": -7, "CT": -5, "DE": -5, "FL": -5, "GA": -5,
    "HI": -10, "ID": -7, "IL": -6, "IN": -5, "IA": -6,
    "KS": -6, "KY": -5, "LA": -6, "ME": -5, "MD": -5,
    "MA": -5, "MI": -5, "MN": -6, "MS": -6, "MO": -6,
    "MT": -7, "NE": -6, "NV": -8, "NH": -5, "NJ": -5,
    "NM": -7, "NY": -5, "NC": -5, "ND": -6, "OH": -5,
    "OK": -6, "OR": -8, "PA": -5, "RI": -5, "SC": -5,
    "SD": -6, "TN": -6, "TX": -6, "UT": -7, "VT": -5,
    "VA": -5, "WA": -8, "WV": -5, "WI": -6, "WY": -7,
    "DC": -5
}
def shift_to_eastern(arr, offset):
    offset = offset
    return np.concatenate([arr[offset:], arr[:offset]])

In [ ]:
# ======
com_fips = set(f.replace("load_", "").replace(".npz", "") for f in os.listdir(com_dir))
res_fips = set(f.replace("load_", "").replace(".npz", "") for f in os.listdir(res_dir))
county_fips_list = sorted(list(com_fips & res_fips))
# ======
state_gen = {}
county_data = {}
for fips in tqdm(county_fips_list, desc="Loading county 8760-hour curves"):
    state = fips[:2]
    com_arr = np.load(os.path.join(com_dir, f"load_{fips}.npz"))
    res_arr = np.load(os.path.join(res_dir, f"load_{fips}.npz"))

    com_total_curve = com_arr['total']  # shape (8760,)
    res_total_curve = res_arr['total']  # shape (8760,)

    county_data[fips] = {
        "com": com_total_curve.copy(),
        "res": res_total_curve.copy()
    }

    if state not in state_gen:
        state_gen[state] = {
            "com": np.zeros(8760),
            "res": np.zeros(8760)
        }

    state_gen[state]["com"] += com_total_curve / 1000.0
    state_gen[state]["res"] += res_total_curve / 1000.0

In [ ]:
state_real = {}
for state in tqdm(state_gen.keys(), desc="Processing EFS real loads"):
    abbr = fips2abbr.get(state)
    if abbr not in state_to_utc_offset:
        print(f"Warning: No timezone info for {state}")
        continue
    offset = state_to_utc_offset[abbr]
    df = efs_df[efs_df["State"] == abbr]
    if df.empty:
        print(f"Warning: No EFS data for {abbr}")
        continue
    load = {
        "com": np.zeros(8760),
        "res": np.zeros(8760),
        "ind+trans": np.zeros(8760)
    }
    for _, row in df.iterrows():
        hour = int(row["LocalHourID"]) - 1
        if 0 <= hour < 8760:
            sector, subsec, val = row["Sector"], row["Subsector"], row["LoadMW"]
            if sector == "Commercial":
                load["com"][hour] += val
            elif sector == "Residential":
                load["res"][hour] += val
            elif sector in ["Industrial", "Transportation"]:
                load["ind+trans"][hour] += val
    state_real[state] = {
        "com": load["com"],         # (8760,)
        "res": load["res"],         # (8760,)
        "ind+trans": load["ind+trans"]  # (8760,)
    }

In [ ]:
for fips in tqdm(county_fips_list, desc="Rescaling and Saving"):
    state = fips[:2]
    if state not in state_gen or state not in state_real:
        continue


    with np.errstate(divide='ignore', invalid='ignore'):
        ratio_com = np.divide(state_real[state]["com"], state_gen[state]["com"],
                              out=np.ones(8760), where=state_gen[state]["com"] != 0)
        ratio_res = np.divide(state_real[state]["res"], state_gen[state]["res"],
                              out=np.ones(8760), where=state_gen[state]["res"] != 0)

    com_arr = np.load(os.path.join(com_dir, f"load_{fips}.npz"))
    res_arr = np.load(os.path.join(res_dir, f"load_{fips}.npz"))

    com_total = com_arr["total"]  # shape (8760,)
    com_heating = com_arr["heating"]
    com_cooling = com_arr["cooling"]

    res_total = res_arr["total"]
    res_heating = res_arr["heating"]
    res_cooling = res_arr["cooling"]

    scaled_com_total = com_total * ratio_com
    scaled_com_heating = com_heating * ratio_com
    scaled_com_cooling = com_cooling * ratio_com

    scaled_res_total = res_total * ratio_res
    scaled_res_heating = res_heating * ratio_res
    scaled_res_cooling = res_cooling * ratio_res

    abbr = fips2abbr[state]
    offset = state_to_utc_offset.get(abbr, 0)

    pop = fips_pop_dict.get(fips, 0)
    state_total_pop = state_pop.get(state, 0)
    ind_trans = np.zeros(8760)
    if state_total_pop > 0:
        ind_trans = state_real[state]["ind+trans"] * (pop / state_total_pop) * 1000  

    scaled_com_total = shift_to_eastern(scaled_com_total, offset)
    scaled_com_heating = shift_to_eastern(scaled_com_heating, offset)
    scaled_com_cooling = shift_to_eastern(scaled_com_cooling, offset)
    scaled_res_total = shift_to_eastern(scaled_res_total, offset)
    scaled_res_heating = shift_to_eastern(scaled_res_heating, offset)
    scaled_res_cooling = shift_to_eastern(scaled_res_cooling, offset)
    ind_trans = shift_to_eastern(ind_trans, offset)

    np.savez_compressed(
        os.path.join(output_dir, f"load_{fips}.npz"),
        com_total=scaled_com_total,
        com_heating=scaled_com_heating,
        com_cooling=scaled_com_cooling,
        res_total=scaled_res_total,
        res_heating=scaled_res_heating,
        res_cooling=scaled_res_cooling,
        ind_trans=ind_trans
    )

In [ ]:
new_dir = "your_directory/loads/new/"

state_real_energy = {}
for state, vals in state_real.items():
    state_real_energy[state] = np.sum(vals["com"]) + np.sum(vals["res"]) + np.sum(vals["ind+trans"]) # GWh

state_new_energy = {}
county_to_state = {}

for file in tqdm(os.listdir(new_dir), desc="Processing new generated loads"):
    if not file.endswith(".npz"):
        continue
    fips = file.replace("load_", "").replace(".npz", "")
    state = fips[:2]
    county_to_state[fips] = state

    data = np.load(os.path.join(new_dir, file))
    com_total = data["com_total"]  # (8760,)
    res_total = data["res_total"]
    ind_total = data["ind_trans"]

    com_energy = np.sum(com_total) / 1000.0  # MW → GWh
    res_energy = np.sum(res_total) / 1000.0
    ind_energy = np.sum(ind_total) / 1000.0

    state_new_energy.setdefault(state, {"com": 0.0, "res": 0.0, "ind": 0.0})
    state_new_energy[state]["com"] += com_energy
    state_new_energy[state]["res"] += res_energy
    state_new_energy[state]["ind"] += ind_energy

def summarize(stateset):
    total = 0.0
    for state, vals in state_new_energy.items():
        if state in stateset:
            total += vals["com"] + vals["res"] + vals["ind"]
    return total

all_states = set(state_new_energy.keys())
states_no_dc = all_states - {"11"} 

new_total_with_dc = summarize(all_states)
new_total_no_dc = summarize(states_no_dc)

real_total_with_dc = sum(state_real_energy.get(state, 0.0) for state in all_states)
real_total_no_dc = sum(state_real_energy.get(state, 0.0) for state in states_no_dc)


print("{:<5} {:>15} {:>15} {:>10}".format("State", "Real_GWh", "New_GWh", "Diff(%)"))
for state in sorted(all_states):
    real = state_real_energy.get(state, 0.0)
    new = state_new_energy.get(state, {}).get("com", 0.0) + state_new_energy.get(state, {}).get("res", 0.0) + state_new_energy.get(state, {}).get("ind", 0.0)
    diff = ((new - real) / real * 100) if real != 0 else 0
    print("{:<5} {:>15.2f} {:>15.2f} {:>10.2f}".format(state, real, new, diff))

print(f"Real Total: {real_total_with_dc:.2f} GWh")
print(f"New Total: {new_total_with_dc:.2f} GWh")
